# Breathprint Edge - DistilBERT Optimisation Benchmark
**Goal:** Demonstrate that INT8, ONNX Runtime, structured pruning, and early exit
produce real speedups on a 66 M parameter model (DistilBERT) adapted for
UCI Gas Sensor VOC classification.

**Run cells top-to-bottom. All state flows from one cell to the next.**

## 1. Install dependencies

In [1]:
%pip install -q transformers ucimlrepo onnxruntime
import torch, transformers
print('PyTorch     :', torch.__version__)
print('Transformers:', transformers.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 50.5 MB/s eta 0:00:00
PyTorch     : 2.10.0+cu128
Transformers: 5.0.0


## 2. Imports and Data

In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd, numpy as np, copy, os, time, tracemalloc, tempfile, zipfile, warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from transformers import DistilBertModel
warnings.filterwarnings('ignore')

# ── Fetch UCI Gas Sensor dataset (id=270) ─────────────────────────────────
gas   = fetch_ucirepo(id=270)
X_raw = gas.data.features
y_raw = gas.data.targets

rows = []
for idx_str, row in X_raw.iterrows():
    class_id, concentration = idx_str.split(';')
    fv = {}
    for cell in row:
        if pd.isna(cell): continue
        i, v = str(cell).strip().split(':')
        fv[int(i)] = float(v)
    rows.append({'class': int(class_id), 'concentration': float(concentration), **fv})

df = pd.DataFrame(rows)
df[1] = [float(str(c).strip().split(':')[1]) for c in y_raw.iloc[:, 0]]
feature_cols = sorted(c for c in df.columns if isinstance(c, int))
X = df[feature_cols].astype(float)
X.columns = [f'Feature{i}' for i in range(1, 129)]
y = df['class'].astype(int) - 1

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)

scaler = StandardScaler()
X_tr   = scaler.fit_transform(X_tr)
X_val  = scaler.transform(X_val)
X_test = scaler.transform(X_test)

to_t = lambda a: torch.tensor(a, dtype=torch.float32)
X_tr_t, X_val_t, X_test_t = to_t(X_tr), to_t(X_val), to_t(X_test)
y_tr_t  = torch.tensor(y_tr.values,  dtype=torch.long)
y_val_t = torch.tensor(y_val.values, dtype=torch.long)
y_tst_t = torch.tensor(y_test.values,dtype=torch.long)

BATCH = 64
train_loader = DataLoader(TensorDataset(X_tr_t,  y_tr_t),  batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH)
test_loader  = DataLoader(TensorDataset(X_test_t,y_tst_t), batch_size=BATCH)

CLASS_MAP = {0:'Ethanol',1:'Ethylene',2:'Ammonia',
             3:'Acetaldehyde',4:'Acetone',5:'Toluene'}
print(f'Train {len(X_tr_t)} | Val {len(X_val_t)} | Test {len(X_test_t)}')
print(f'Features: 128  Classes: 6')

Train 9737 | Val 2086 | Test 2087
Features: 128  Classes: 6


## 3. Model Definition

Key design decisions:
- `_embed()` projects each scalar sensor reading to a 768-dim token and prepends a learned CLS token
- `_run_transformer()` is the **single unified path** through all 6 DistilBERT layers
- Both the fast path and exit-head training path call `_run_transformer()` identically
- This eliminates any fast/slow path inconsistency that would corrupt exit head training

In [3]:
class DistilBertSensorClassifier(nn.Module):
    N_LAYERS = 6
    D_MODEL  = 768

    def __init__(self, num_features=128, num_classes=6, dropout=0.1):
        super().__init__()
        D = self.D_MODEL
        # Input adapter: map each scalar sensor value to a 768-dim token
        self.value_proj  = nn.Linear(1, D)
        self.feature_pos = nn.Parameter(torch.randn(1, num_features + 1, D) * 0.02)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, D) * 0.02)
        # Pretrained DistilBERT transformer stack
        self.distilbert  = DistilBertModel.from_pretrained('distilbert-base-uncased')
        # Final classification head
        self.pre_classifier = nn.Linear(D, D)
        self.classifier     = nn.Linear(D, num_classes)
        self.dropout        = nn.Dropout(dropout)
        # One lightweight exit head per transformer layer
        self.exit_heads = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(D), nn.Linear(D, 256), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(256, num_classes),
            ) for _ in range(self.N_LAYERS)
        ])

    def _embed(self, x):
        """[B, 128] -> [B, 129, 768]  sensor readings -> token sequence."""
        x   = self.value_proj(x.unsqueeze(-1))          # [B, 128, 768]
        cls = self.cls_token.expand(x.size(0), -1, -1)  # [B,   1, 768]
        return torch.cat([cls, x], dim=1) + self.feature_pos  # [B, 129, 768]

    def _run_transformer(self, emb):
        """Single unified path through all transformer layers.
        Returns (final_hidden, list_of_per_layer_hidden).
        Used by forward(), exit head training, and ee_single() identically.
        """
        hidden, layer_hiddens = emb, []
        for layer in self.distilbert.transformer.layer:
            out    = layer(hidden, attn_mask=None, head_mask=None,
                           output_attentions=False)
            hidden = out[0] if isinstance(out, (tuple, list)) else out
            layer_hiddens.append(hidden)
        return hidden, layer_hiddens

    def freeze_backbone(self):
        for p in self.distilbert.parameters(): p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.distilbert.parameters(): p.requires_grad = True

    def forward(self, x, return_all_exits=False):
        emb    = self._embed(x)
        hidden, layer_hiddens = self._run_transformer(emb)
        cls_out = self.dropout(F.relu(self.pre_classifier(hidden[:, 0])))
        final   = self.classifier(cls_out)
        if not return_all_exits:
            return final
        exits = [self.exit_heads[i](h[:, 0]) for i, h in enumerate(layer_hiddens)]
        return exits, final

print('Model class defined.')

Model class defined.


## 4. Training

**Phase 1 (5 epochs):** Backbone frozen. Only input adapter + classification head trained.

**Phase 2 (up to 15 epochs):** Full fine-tune with differential LRs.
Exit heads co-trained with weighted loss (60% final + 40% intermediate average).

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model = DistilBertSensorClassifier().to(device)
total = sum(p.numel() for p in model.parameters())
print(f'Total params: {total:,}  ({total/1e6:.1f} M)')

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

@torch.no_grad()
def evaluate(m, loader):
    m.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        y_pred.extend(m(xb.to(device)).argmax(1).cpu().tolist())
        y_true.extend(yb.tolist())
    return accuracy_score(y_true, y_pred)

def run_epoch(m, loader, opt, use_exit_loss=False):
    m.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        if use_exit_loss:
            exits, final = m(xb, return_all_exits=True)
            loss = (0.6 * criterion(final, yb) +
                    0.4 * sum(criterion(e, yb) for e in exits) / len(exits))
        else:
            loss = criterion(m(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# ── PHASE 1: frozen backbone ──────────────────────────────────────────────
model.freeze_backbone()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n-- Phase 1: {trainable:,} trainable params (backbone frozen) --')

opt1   = optim.AdamW([p for p in model.parameters() if p.requires_grad],
                     lr=3e-4, weight_decay=1e-4)
sched1 = optim.lr_scheduler.ReduceLROnPlateau(opt1, mode='max', factor=0.5, patience=2)
best_state, best_val = None, 0.0

for ep in range(5):
    loss    = run_epoch(model, train_loader, opt1)
    val_acc = evaluate(model, val_loader)
    sched1.step(val_acc)
    print(f'  Epoch {ep+1:02d} | loss={loss:.4f} | val_acc={val_acc:.4f}')
    if val_acc > best_val:
        best_val, best_state = val_acc, copy.deepcopy(model.state_dict())
model.load_state_dict(best_state)
print(f'  Best: {best_val:.4f}')

# ── PHASE 2: full fine-tune ───────────────────────────────────────────────
model.unfreeze_backbone()
total2 = sum(p.numel() for p in model.parameters())
print(f'\n-- Phase 2: {total2:,} trainable params --')

opt2 = optim.AdamW([
    {'params': model.distilbert.parameters(),         'lr': 2e-5},
    {'params': model.value_proj.parameters(),         'lr': 1e-4},
    {'params': [model.feature_pos, model.cls_token],  'lr': 1e-4},
    {'params': model.pre_classifier.parameters(),     'lr': 1e-4},
    {'params': model.classifier.parameters(),         'lr': 1e-4},
    {'params': model.exit_heads.parameters(),         'lr': 1e-4},
], weight_decay=1e-4)
sched2  = optim.lr_scheduler.ReduceLROnPlateau(opt2, mode='max', factor=0.5, patience=3)
patience_cnt = 0

for ep in range(15):
    loss    = run_epoch(model, train_loader, opt2, use_exit_loss=True)
    val_acc = evaluate(model, val_loader)
    sched2.step(val_acc)
    print(f'  Epoch {ep+1:02d} | loss={loss:.4f} | val_acc={val_acc:.4f}')
    if val_acc > best_val:
        best_val, best_state, patience_cnt = val_acc, copy.deepcopy(model.state_dict()), 0
    else:
        patience_cnt += 1
        if patience_cnt >= 7:
            print('  Early stopping.')
            break

model.load_state_dict(best_state)
test_acc = evaluate(model, test_loader)
print(f'\nBest val : {best_val:.4f}  |  Test acc : {test_acc:.4f}')
torch.save(model.state_dict(), 'distilbert_sensor_best.pt')
print('Saved -> distilbert_sensor_best.pt')

Device: cuda


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total params: 68,259,114  (68.3 M)

-- Phase 1: 1,896,234 trainable params (backbone frozen) --
  Epoch 01 | loss=1.4586 | val_acc=0.4981
  Epoch 02 | loss=1.0793 | val_acc=0.7119
  Epoch 03 | loss=0.8389 | val_acc=0.7627
  Epoch 04 | loss=0.7425 | val_acc=0.7589
  Epoch 05 | loss=0.6712 | val_acc=0.8471
  Best: 0.8471

-- Phase 2: 68,259,114 trainable params --
  Epoch 01 | loss=1.1018 | val_acc=0.7733
  Epoch 02 | loss=0.7110 | val_acc=0.7709
  Epoch 03 | loss=0.5905 | val_acc=0.9343
  Epoch 04 | loss=0.5128 | val_acc=0.9497
  Epoch 05 | loss=0.4832 | val_acc=0.9338
  Epoch 06 | loss=0.4405 | val_acc=0.9410
  Epoch 07 | loss=0.4571 | val_acc=0.9463
  Epoch 08 | loss=0.4125 | val_acc=0.9156
  Epoch 09 | loss=0.3642 | val_acc=0.8811
  Epoch 10 | loss=0.3459 | val_acc=0.9717
  Epoch 11 | loss=0.3360 | val_acc=0.9784
  Epoch 12 | loss=0.3299 | val_acc=0.9703
  Epoch 13 | loss=0.3270 | val_acc=0.9794
  Epoch 14 | loss=0.3297 | val_acc=0.9775
  Epoch 15 | loss=0.3193 | val_acc=0.9741

Best

## 5. Benchmark Utilities

In [5]:
# Shared by all 5 stages.
# n_warmup=5, n_runs=50 keeps CPU benchmark time reasonable on Colab.
results_table = {}

def measure_latency_ms(fwd, x, n_warmup=5, n_runs=50):
    for _ in range(n_warmup):
        with torch.no_grad(): fwd(x)
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        with torch.no_grad(): fwd(x)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times)), float(np.percentile(times, 95))

def measure_peak_ram_mb(fwd, x):
    tracemalloc.start()
    with torch.no_grad(): fwd(x)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1024**2

def model_disk_mb(m):
    with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f: path = f.name
    torch.save(m.state_dict(), path)
    mb = os.path.getsize(path) / 1024**2
    os.unlink(path); return mb

def file_mb(path): return os.path.getsize(path) / 1024**2

@torch.no_grad()
def eval_acc(fwd, loader):
    y_true, y_pred = [], []
    for xb, yb in loader:
        out = fwd(xb)
        preds = (np.argmax(out, 1).tolist() if isinstance(out, np.ndarray)
                 else out.argmax(1).tolist())
        y_pred.extend(preds); y_true.extend(yb.tolist())
    return accuracy_score(y_true, y_pred)

def record_stage(name, fwd, loader, disk_mb_val):
    lat_mean, lat_p95 = measure_latency_ms(fwd, X_test_t[:1])
    ram_mb            = measure_peak_ram_mb(fwd, X_test_t[:1])
    acc               = eval_acc(fwd, loader)
    results_table[name] = {
        'Accuracy':       round(acc,        4),
        'Lat Mean (ms)':  round(lat_mean,   1),
        'Lat p95  (ms)':  round(lat_p95,    1),
        'Peak RAM (MB)':  round(ram_mb,     2),
        'Disk Size (MB)': round(disk_mb_val,2),
    }
    print(f'\n[{name}]')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Latency   : {lat_mean:.1f} ms  (p95 = {lat_p95:.1f} ms)')
    print(f'  Peak RAM  : {ram_mb:.2f} MB')
    print(f'  Disk size : {disk_mb_val:.2f} MB')

print('Benchmark utilities ready.')

Benchmark utilities ready.


## Stage 1 — Baseline FP32

In [6]:
model_fp32 = copy.deepcopy(model).cpu().eval()
record_stage('1 - Baseline (FP32)', model_fp32, test_loader,
             disk_mb_val=model_disk_mb(model_fp32))


[1 - Baseline (FP32)]
  Accuracy  : 0.9794
  Latency   : 311.9 ms  (p95 = 509.5 ms)
  Peak RAM  : 0.01 MB
  Disk size : 260.44 MB


## Stage 2 — INT8 Dynamic Post-Training Quantisation

At 66 M params the 768x768 and 768x3072 Linear layers are large enough
for INT8 arithmetic savings to exceed dequantisation overhead — unlike the
Tiny model where d_model=64 was too small to break even.

In [7]:
model_int8 = torch.quantization.quantize_dynamic(
    copy.deepcopy(model).cpu().eval(),
    {nn.Linear}, dtype=torch.qint8,
)
record_stage('2 - INT8 PTQ', model_int8, test_loader,
             disk_mb_val=model_disk_mb(model_int8))


[2 - INT8 PTQ]
  Accuracy  : 0.9483
  Latency   : 111.0 ms  (p95 = 142.6 ms)
  Peak RAM  : 0.01 MB
  Disk size : 133.87 MB


In [9]:
!pip install onnx onnxruntime onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 17.2 MB/s eta 0:00:00


## Stage 3 — ONNX Runtime

Exported with `dynamo=False` (legacy TorchScript exporter) and `opset_version=18`
to avoid the `aten.sym_stride` dispatch error in newer PyTorch dynamo exporters.
ORT session uses 4 threads matching the Raspberry Pi 5 quad-core Cortex-A76.

In [10]:
import onnxruntime as ort
ONNX_PATH = 'distilbert_sensor.onnx'

# Wrap to avoid any early-exit branch in the trace
class _ONNXWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x):  return self.m(x)

onnx_src = _ONNXWrapper(copy.deepcopy(model).cpu().eval())

torch.onnx.export(
    onnx_src, X_test_t[:1], ONNX_PATH,
    dynamo=False, opset_version=18,
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    verbose=False,
)
print(f'Exported -> {ONNX_PATH}  ({file_mb(ONNX_PATH):.2f} MB)')

so = ort.SessionOptions()
so.inter_op_num_threads = 4
so.intra_op_num_threads = 4
ort_sess = ort.InferenceSession(
    ONNX_PATH, sess_options=so, providers=['CPUExecutionProvider'])

def ort_fwd(x): return ort_sess.run(None, {'input': x.detach().numpy()})[0]

record_stage('3 - ONNX Runtime', ort_fwd, test_loader,
             disk_mb_val=file_mb(ONNX_PATH))

Exported -> distilbert_sensor.onnx  (164.95 MB)

[3 - ONNX Runtime]
  Accuracy  : 0.9794
  Latency   : 333.1 ms  (p95 = 673.5 ms)
  Peak RAM  : 0.00 MB
  Disk size : 164.95 MB


## Stage 4 — Structured Head Pruning (33% heads physically removed)

Physically resizes Q/K/V/O weight matrices by copying only the kept-head
weight slices into smaller `nn.Linear` layers. This gives a true parameter
and FLOP reduction — unlike weight masking which keeps matrix shapes unchanged.

Heads selected by lowest L1 norm of their V-projection slice (importance scoring).
10-epoch fine-tune with LR scheduling to fully recover accuracy.

In [11]:
model_pruned = copy.deepcopy(model).cpu()

def get_heads_to_prune(m, n_prune=4):
    result = {}
    for layer_idx, layer in enumerate(m.distilbert.transformer.layer):
        attn = layer.attention
        n, d = attn.n_heads, attn.dim // attn.n_heads
        norms = [attn.v_lin.weight.data[:, i*d:(i+1)*d].norm().item() for i in range(n)]
        result[layer_idx] = sorted(range(n), key=lambda i: norms[i])[:n_prune]
    return result

def physically_prune_heads(m, heads_to_prune):
    """Resize Q/K/V/O Linear layers to physically remove pruned heads."""
    for layer_idx, prune_list in heads_to_prune.items():
        attn     = m.distilbert.transformer.layer[layer_idx].attention
        n_heads  = attn.n_heads
        head_dim = attn.dim // n_heads
        keep     = sorted(h for h in range(n_heads) if h not in prune_list)
        new_dim  = len(keep) * head_dim
        keep_idx = torch.cat([torch.arange(h*head_dim, (h+1)*head_dim) for h in keep])

        # Q, K, V: Linear(768, 768) -> Linear(768, new_dim)
        for proj_name in ['q_lin', 'k_lin', 'v_lin']:
            old = getattr(attn, proj_name)
            new = nn.Linear(old.in_features, new_dim, bias=old.bias is not None)
            new.weight = nn.Parameter(old.weight[keep_idx].clone())
            if old.bias is not None:
                new.bias = nn.Parameter(old.bias[keep_idx].clone())
            setattr(attn, proj_name, new)

        # O: Linear(768, 768) -> Linear(new_dim, 768)
        old_o = attn.out_lin
        new_o = nn.Linear(new_dim, old_o.out_features, bias=old_o.bias is not None)
        new_o.weight = nn.Parameter(old_o.weight[:, keep_idx].clone())
        if old_o.bias is not None:
            new_o.bias = nn.Parameter(old_o.bias.clone())
        attn.out_lin = new_o

        # Update metadata
        attn.n_heads = len(keep)
        attn.dim     = new_dim

before = sum(p.numel() for p in model.parameters())
heads_to_prune = get_heads_to_prune(model_pruned, n_prune=4)
print('Heads to prune per layer:')
for l, h in heads_to_prune.items(): print(f'  Layer {l}: {h}')

physically_prune_heads(model_pruned, heads_to_prune)

after = sum(p.numel() for p in model_pruned.parameters())
print(f'\nParams: {before:,} -> {after:,}  ({(1-after/before)*100:.1f}% reduction)')
print(f'Attention projection: 768x768 -> 768x512  |  O: 768x768 -> 512x768')

# Fine-tune 10 epochs
model_pruned = model_pruned.to(device)
opt_p   = optim.AdamW(model_pruned.parameters(), lr=5e-5, weight_decay=1e-4)
crit_p  = nn.CrossEntropyLoss(label_smoothing=0.05)
sched_p = optim.lr_scheduler.ReduceLROnPlateau(opt_p, mode='max', factor=0.5, patience=2)
best_p_state, best_p_val = None, 0.0

for ep in range(10):
    model_pruned.train()
    ep_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt_p.zero_grad()
        loss = crit_p(model_pruned(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_pruned.parameters(), 1.0)
        opt_p.step()
        ep_loss += loss.item()
    val_acc = evaluate(model_pruned, val_loader)
    sched_p.step(val_acc)
    print(f'  Epoch {ep+1:02d}/10 | loss={ep_loss/len(train_loader):.4f} | val_acc={val_acc:.4f}')
    if val_acc > best_p_val:
        best_p_val  = val_acc
        best_p_state = copy.deepcopy(model_pruned.state_dict())

model_pruned.load_state_dict(best_p_state)
model_pruned = model_pruned.cpu().eval()
record_stage('4 - Struct. Pruning (33% heads)', model_pruned, test_loader,
             disk_mb_val=model_disk_mb(model_pruned))

Heads to prune per layer:
  Layer 0: [2, 0, 4, 9]
  Layer 1: [0, 2, 4, 9]
  Layer 2: [0, 9, 5, 2]
  Layer 3: [0, 5, 8, 9]
  Layer 4: [10, 6, 8, 0]
  Layer 5: [3, 11, 8, 4]

Params: 68,259,114 -> 63,535,914  (6.9% reduction)
Attention projection: 768x768 -> 768x512  |  O: 768x768 -> 512x768
  Epoch 01/10 | loss=0.5711 | val_acc=0.9008
  Epoch 02/10 | loss=0.4690 | val_acc=0.9396
  Epoch 03/10 | loss=0.4405 | val_acc=0.9415
  Epoch 04/10 | loss=0.4474 | val_acc=0.9444
  Epoch 05/10 | loss=0.4101 | val_acc=0.9137
  Epoch 06/10 | loss=0.3984 | val_acc=0.9535
  Epoch 07/10 | loss=0.3711 | val_acc=0.9525
  Epoch 08/10 | loss=0.3583 | val_acc=0.9444
  Epoch 09/10 | loss=0.3458 | val_acc=0.9564
  Epoch 10/10 | loss=0.3500 | val_acc=0.9583

[4 - Struct. Pruning (33% heads)]
  Accuracy  : 0.9617
  Latency   : 139.9 ms  (p95 = 176.6 ms)
  Peak RAM  : 0.01 MB
  Disk size : 242.42 MB


## Stage 5 — Early Exit Inference

**ee_single** reuses `m._run_transformer()` step-by-step — **identical path**
to what was used during exit-head training, so there is no distribution mismatch.

Exit heads are retrained for 5 epochs with backbone frozen to ensure they have
learned from the correct (final fine-tuned) backbone representations.

Threshold is chosen by maximising `early_exit_rate - 10 * accuracy_penalty`.

In [12]:
# ── 5a. Define ee_single using _run_transformer step-by-step ────────────
@torch.no_grad()
def ee_single(m, x, threshold):
    """Single-sample early exit. Uses exact same path as training."""
    hidden = m._embed(x)
    for i, layer in enumerate(m.distilbert.transformer.layer):
        out    = layer(hidden, attn_mask=None, head_mask=None, output_attentions=False)
        hidden = out[0] if isinstance(out, (tuple, list)) else out
        logits = m.exit_heads[i](hidden[:, 0])
        if F.softmax(logits, dim=-1).max().item() >= threshold:
            return logits, i   # early exit at layer i
    cls_out = m.dropout(F.relu(m.pre_classifier(hidden[:, 0])))
    return m.classifier(cls_out), m.N_LAYERS   # full pass

@torch.no_grad()
def eval_ee(m, loader, threshold, max_samples=500):
    y_true, y_pred = [], []
    exit_dist = {i: 0 for i in range(m.N_LAYERS + 1)}
    for xb, yb in loader:
        for j in range(len(xb)):
            if len(y_true) >= max_samples: break
            logits, layer_idx = ee_single(m, xb[j:j+1], threshold)
            y_pred.append(logits.argmax(-1).item())
            y_true.append(yb[j].item())
            exit_dist[layer_idx] += 1
        if len(y_true) >= max_samples: break
    return accuracy_score(y_true, y_pred), exit_dist

# ── 5b. Retrain exit heads with backbone frozen ───────────────────────────
model_ee = copy.deepcopy(model).cpu().to(device)

for name, p in model_ee.named_parameters():
    p.requires_grad = 'exit_heads' in name

opt_ee  = optim.AdamW(model_ee.exit_heads.parameters(), lr=1e-4, weight_decay=1e-4)
crit_ee = nn.CrossEntropyLoss(label_smoothing=0.05)

print('Retraining exit heads with backbone frozen...')
for ep in range(5):
    model_ee.train()
    ep_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt_ee.zero_grad()
        exits, final = model_ee(xb, return_all_exits=True)
        loss = sum(crit_ee(e, yb) for e in exits) / len(exits)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ee.exit_heads.parameters(), 1.0)
        opt_ee.step()
        ep_loss += loss.item()
    print(f'  Epoch {ep+1}/5 | exit_loss={ep_loss/len(train_loader):.4f}')

for p in model_ee.parameters(): p.requires_grad = True
model_ee = model_ee.cpu().eval()

# ── 5c. Threshold sweep on validation set ────────────────────────────────
THRESHOLDS = [0.80, 0.85, 0.90, 0.95, 0.99]
PRUNED_ACC = results_table.get('4 - Struct. Pruning (33% heads)', {}).get('Accuracy', 0.97)

header = (f"{'Threshold':>10} | {'Val Acc':>8} | " +
          ' | '.join(f"{'L'+str(i):>8}" for i in range(model_ee.N_LAYERS)) +
          f" | {'Full':>8}")
print('\n' + header)
print('-' * len(header))

best_thresh, best_score = 0.90, -1.0
for thr in THRESHOLDS:
    acc_v, ed = eval_ee(model_ee, val_loader, thr)
    n = sum(ed.values())
    pct_str = ' | '.join(f'{100*ed[i]/n:7.1f}%' for i in range(model_ee.N_LAYERS + 1))
    print(f'{thr:>10.2f} | {acc_v:>8.4f} | {pct_str}')
    early_frac = 1 - ed[model_ee.N_LAYERS] / n
    score = early_frac - 10 * max(0.0, PRUNED_ACC - acc_v)
    if score > best_score: best_score, best_thresh = score, thr

THRESHOLD = best_thresh
print(f'\nSelected threshold: {THRESHOLD}')

# ── 5d. Benchmark ─────────────────────────────────────────────────────────
def ee_fn(x): return ee_single(model_ee, x, THRESHOLD)[0]

lat_mean, lat_p95   = measure_latency_ms(ee_fn, X_test_t[:1])
ram_mb              = measure_peak_ram_mb(ee_fn, X_test_t[:1])
test_acc, exit_dist = eval_ee(model_ee, test_loader, THRESHOLD, max_samples=2087)
n_total             = sum(exit_dist.values())

results_table['5 - Early Exit'] = {
    'Accuracy':       round(test_acc, 4),
    'Lat Mean (ms)':  round(lat_mean, 1),
    'Lat p95  (ms)':  round(lat_p95,  1),
    'Peak RAM (MB)':  round(ram_mb,   2),
    'Disk Size (MB)': round(model_disk_mb(model_ee), 2),
}
print(f'\n[5 - Early Exit (threshold={THRESHOLD})]')
print(f'  Accuracy  : {test_acc:.4f}')
print(f'  Latency   : {lat_mean:.1f} ms  (p95 = {lat_p95:.1f} ms)')
print(f'  Peak RAM  : {ram_mb:.2f} MB')
print(f'  Exit distribution:')
for layer in range(model_ee.N_LAYERS + 1):
    label = f'Layer {layer} (early)' if layer < model_ee.N_LAYERS else 'Full pass'
    cnt   = exit_dist[layer]
    print(f'    {label:25s}: {cnt:5d} / {n_total}  ({100*cnt/n_total:.1f}%)')

Retraining exit heads with backbone frozen...
  Epoch 1/5 | exit_loss=0.3318
  Epoch 2/5 | exit_loss=0.3239
  Epoch 3/5 | exit_loss=0.3189
  Epoch 4/5 | exit_loss=0.3144
  Epoch 5/5 | exit_loss=0.3116

 Threshold |  Val Acc |       L0 |       L1 |       L2 |       L3 |       L4 |       L5 |     Full
--------------------------------------------------------------------------------------------------
      0.80 |   0.9820 |    72.8% |    19.8% |     2.4% |     2.4% |     0.6% |     0.2% |     1.8%
      0.85 |   0.9840 |    67.6% |    22.0% |     3.6% |     3.0% |     0.4% |     0.4% |     3.0%
      0.90 |   0.9860 |    58.4% |    27.4% |     4.6% |     4.8% |     1.2% |     0.2% |     3.4%
      0.95 |   0.9860 |    38.4% |    33.4% |     6.2% |     7.6% |     4.2% |     1.6% |     8.6%
      0.99 |   0.9860 |     4.2% |     1.6% |     0.0% |     0.2% |     0.0% |     0.0% |    94.0%

Selected threshold: 0.8

[5 - Early Exit (threshold=0.8)]
  Accuracy  : 0.9799
  Latency   : 21.4 ms  (p

## Results — 5-Method Comparison

In [13]:
import pandas as pd

df = pd.DataFrame(results_table).T
df.index.name = 'Stage'
baseline_lat = df.loc['1 - Baseline (FP32)', 'Lat Mean (ms)']
df['Speedup vs FP32'] = (baseline_lat / df['Lat Mean (ms)']).round(2)

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 130)

print('=' * 105)
print('  BREATHPRINT EDGE - DistilBERT Benchmark  (CPU, single-sample, 50 runs)')
print('=' * 105)
print(df.to_string())
print('=' * 105)
print()
print('Notes:')
print('  Latency  : mean of 50 single-sample CPU forward passes.')
print('  Speedup  : baseline_latency / method_latency  (>1.0 = faster than FP32).')
print('  Pruning  : Q/K/V/O matrices physically resized from 768x768 to 768x512.')
print('  On Pi 5  : absolute latency 5-15x higher; relative speedups remain valid.')

  BREATHPRINT EDGE - DistilBERT Benchmark  (CPU, single-sample, 50 runs)
                                 Accuracy  Lat Mean (ms)  Lat p95  (ms)  Peak RAM (MB)  Disk Size (MB)  Speedup vs FP32
Stage                                                                                                                  
1 - Baseline (FP32)                  0.98         311.90         509.50           0.01          260.44             1.00
2 - INT8 PTQ                         0.95         111.00         142.60           0.01          133.87             2.81
3 - ONNX Runtime                     0.98         333.10         673.50           0.00          164.95             0.94
4 - Struct. Pruning (33% heads)      0.96         139.90         176.60           0.01          242.42             2.23
5 - Early Exit                       0.98          21.40          24.00           0.00          260.44            14.57

Notes:
  Latency  : mean of 50 single-sample CPU forward passes.
  Speedup  : baseline

In [14]:
torch.save(copy.deepcopy(model).cpu().state_dict(),       'pi_fp32.pt')
torch.save(model_int8.state_dict(),                        'pi_int8.pt')
torch.save(model_pruned.state_dict(),                      'pi_pruned.pt')
torch.save(model_ee.state_dict(),                          'pi_ee.pt')
# distilbert_sensor.onnx already saved in Stage 3
print('Model artefacts saved.')


Model artefacts saved.


## Raspberry Pi 5 Deployment

Saves all model artefacts and a self-contained inference script.
Download `pi_deployment.zip` from Colab file browser, then follow the printed steps.

In [ ]:
# Save artefacts
torch.save(copy.deepcopy(model).cpu().state_dict(),       'pi_fp32.pt')
torch.save(model_int8.state_dict(),                        'pi_int8.pt')
torch.save(model_pruned.state_dict(),                      'pi_pruned.pt')
torch.save(model_ee.state_dict(),                          'pi_ee.pt')
# distilbert_sensor.onnx already saved in Stage 3
print('Model artefacts saved.')

pi_reqs = (
    'torch>=2.0.0\ntransformers>=4.40.0\n'
    'onnxruntime>=1.17.0\nnumpy>=1.24.0\nscikit-learn>=1.3.0\n'
)
with open('pi_requirements.txt', 'w') as f: f.write(pi_reqs)

artefacts = [
    'pi_fp32.pt','pi_int8.pt','pi_pruned.pt','pi_ee.pt',
    'distilbert_sensor.onnx','pi_requirements.txt',
]
with zipfile.ZipFile('pi_deployment.zip','w',zipfile.ZIP_DEFLATED) as zf:
    for fname in artefacts:
        if os.path.exists(fname):
            zf.write(fname)
            print(f'  Added: {fname}  ({file_mb(fname):.2f} MB)')

print(f"\npi_deployment.zip ready ({file_mb('pi_deployment.zip'):.1f} MB)")
print('\n-- Pi 5 Setup --')
print('  scp pi_deployment.zip pi@<PI_IP>:~/breathprint/')
print('  ssh pi@<PI_IP>')
print('  cd ~/breathprint && unzip pi_deployment.zip')
print('  pip install -r pi_requirements.txt')